# Example 1: 2D Native-Invasive Model with optimal control

$$
\frac{∂N_1}{∂t} = D_1\frac{∂^2N_1}{∂x^2} - r_1\frac{∂N_1}{∂x}  + \left(\theta_1(t, u(x, t)) - a_{11} N_1 \right) N_1 - a_{12}N_1N_2 ,
$$
$$
\frac{∂N_2}{∂t} = D_2\frac{∂^2N_2}{∂x^2} - r_2\frac{∂N_2}{∂x}  + \left(\theta_2(t, u(x, t)) - a_{22} N_2 \right) N_2 - a_{21}N_1N_2
$$

According to the paper, here are the value of each parameters. $D_1, D_2 = 0.05, 0.01, r_1, r_2 = 0.2, 0.1, a_{11} = 0.005, a_{12} = 0.001, a_{21} = 0.002, a_{22} = 0.0012$.

The control influences the growth rate $\theta_1(t, u(x, t))$ and $\theta_2(t, u(x, t))$ as follows:
$$
\theta_1(t, u(x, t)) = a_1 u(x, t)^2 + b_1 u(x, t) + c_1 \\
$$
$$
\theta_2(t, u(x, t)) = a_2 u(x, t)^2 + b_2 u(x, t) + c_2 \\
$$

Where $a_i, b_i, c_i, i=1, 2$ are constant. Then, $u(x, t)$ could be a function or constant. For example if the control is flooding:  
- $u(x, 0) = 1$ represent moderate flooding  
- $u(x, 0) = 0$ represent no flooding

If $u(x, t)$ is a function, could be represent such as:
$$
u(x, t) = \sin \left( \frac{\pi x}{L} \right) e^{\left( \frac{-t}{T}\right)} 
$$


1. Initial conditions (according to paper):
   $$
   N_1(x, 0) = x(1-x)
   $$
   $$
   N_2(x, 0) = 3x(1-x)
   $$

1. At the left boundary $(x = 0)$:
   $$
   N_1(0, t)=N_2(0, t) = 0
   $$

2. At the right boundary $(x = L)$:
   $$
   N_1(L_x, t)=N_2(L_x, t) = 0
   $$
   
Paper references:
OPTIMAL CONTROL APPLIED TO NATIVE-INVASIVE SPECIES COMPETITION VIA A PDE MODEL

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

# Define parameters
L = 1.0       # Spatial domain length
T = 5.0       # Total simulation time
Nx = 200      # Number of spatial points
Nt = 200      # Number of time steps
dx = L / Nx   # Spatial step
dt = T / Nt   # Time step

D1 = 0.05  # Diffusion coefficient for N1
D2 = 0.01  # Diffusion coefficient for N2
r1 = 0.1   # Growth rate for N1
r2 = 0.1   # Growth rate for N2
a11, a12, a21, a22 = 0.005, 0.001, 0.12, 0.02

# Define time intervals where control u is active
intervals = []

def X_tau(t, intervals):
    X_tau_values = np.zeros_like(t, dtype=np.float32)
    for (t_start, t_end) in intervals:
        X_tau_values += np.where((t >= t_start) & (t <= t_end), 1.0, 0.0)
    return np.clip(X_tau_values, 0.0, 1.0)
    
# Growth rate modification functions
def theta1(u):
    return 0.1 * u**2 + 1 * u + 0.2

def theta2(u):
    return -0.1 * u**2 - 1.0 * u + 0.4

# Define ODE function
def pde_system(Y, t, Nx, dx, D1, D2, r1, r2, a11, a12, a21, a22):
    u = X_tau(t, intervals)
    N1, N2 = Y[:Nx], Y[Nx:]
    dN1dt = np.zeros(Nx)
    dN2dt = np.zeros(Nx)
    
    # Apply finite difference method (FDM)
    for i in range(1, Nx - 1):
        dN1dt[i] = (D1 * (N1[i+1] - 2 * N1[i] + N1[i-1]) / dx**2 -
                    r1 * (N1[i+1] - N1[i-1]) / (2*dx) +
                    (theta1(u) - a11 * N1[i]) * N1[i] - a12 * N1[i] * N2[i])
        dN2dt[i] = (D2 * (N2[i+1] - 2 * N2[i] + N2[i-1]) / dx**2 -
                    r2 * (N2[i+1] - N2[i-1]) / (2*dx) +
                    (theta2(u) - a22 * N2[i]) * N2[i] - a21 * N1[i] * N2[i])
    
    # Boundary conditions (Dirichlet)
    dN1dt[0] = dN1dt[-1] = 0
    dN2dt[0] = dN2dt[-1] = 0
    
    return np.concatenate([dN1dt, dN2dt])

# Initial conditions
x = np.linspace(0, L, Nx)
N1_init = x * (1 - x)
N2_init = (1/2) * x * (1 - x) 
Y0 = np.concatenate([N1_init, N2_init])

# Solve using odeint
t = np.linspace(0, T, Nt)
solution = odeint(pde_system, Y0, t, args=(Nx, dx, D1, D2, r1, r2, a11, a12, a21, a22))

# Extract solutions
fdm_N1_pred = solution[:, :Nx].T
fdm_N2_pred = solution[:, Nx:].T

# Create space-time grids
t_grid, x_grid = np.meshgrid(t, x)

# Plot species N1 and N2
fig = plt.figure(figsize=[10, 5])

vmin = min(fdm_N1_pred.min(), fdm_N2_pred.min())
vmax = max(fdm_N1_pred.max(), fdm_N2_pred.max())

ax1 = fig.add_subplot(121)
cax = ax1.contourf(x_grid, t_grid, fdm_N1_pred, 50, cmap='jet', vmin=vmin, vmax=vmax)
ax1.set_xlabel('Spatial coordinate (x)')
ax1.set_ylabel('Time (t)')
ax1.set_title('Population Density of Native Species $N_1$')
fig.colorbar(cax, ax=ax1, location='right')

ax2 = fig.add_subplot(122)
cbx = ax2.contourf(x_grid, t_grid, fdm_N2_pred, 50, cmap='jet', vmin=vmin, vmax=vmax)
ax2.set_xlabel('Spatial coordinate (x)')
ax2.set_title('Population Density of Invasive Species $N_2$')
fig.colorbar(cbx, ax=ax2, location='right')

plt.show()

In [ ]:
# Plot species N1 and N2 at certain time steps
fig = plt.figure(figsize=(9, 3))
ax1 = fig.add_subplot(121)
ax1.set_facecolor('lightgrey')
ax1.grid(which='major', color='black', linestyle='--', linewidth=0.5)
ax1.plot(x, fdm_N1_pred[:, 0], label='t=%.2f' % (t[0]))
ax1.set_ylim([0, 0.5])
ax1.set_xlabel('Spatial coordinate ($x$)', fontsize=12, fontweight ='bold')
ax1.set_ylabel('Population density', fontsize=12, fontweight ='bold')
ax1.legend(title='Time (t)', ncol=2)
ax1.set_title('Population density species $N_1$')

ax2 = fig.add_subplot(122)
ax2.set_facecolor('lightgrey')
ax2.grid(which='major', color='black', linestyle='--', linewidth=0.5)
ax2.plot(x, fdm_N2_pred[:, 0], label='t=%.2f' % (t[0]))
ax2.set_ylim([0, 0.5])
ax2.set_xlabel('Spatial coordinate ($x$)', fontsize=12, fontweight ='bold')
ax2.set_ylabel('Population density', fontsize=12, fontweight ='bold')
ax2.legend(title='Time (t)', ncol=2)
ax2.set_title('Population density species $N_2$')
plt.show()


In [ ]:
def get_parameters():
    D1, D2 = 0.05, 0.01
    r1, r2 = 0.1, 0.1
    a11, a12, a21, a22 = 0.005, 0.001, 0.12, 0.02
    return D1, D2, r1, r2, a11, a12, a21, a22

def get_theta(X_tau):
    u = 1
    a1, b1, c1 = 0.1, 1.0, 0.2
    a2, b2, c2 = -0.1, -1.0, 0.4
    theta_1 = (a1*u**2 + b1*u)*X_tau + c1
    theta_2 = (a2*u**2 + b2*u)*X_tau + c2
    return theta_1, theta_2

def X_tau(t, intervals):
    X_tau_values = tf.zeros_like(t, dtype=tf.float32)
    for (t_start, t_end) in intervals:
        X_tau_values += tf.where((t >= t_start) & (t <= t_end), 1.0, 0.0)
    
    X_tau_values = tf.clip_by_value(X_tau_values, 0.0, 1.0)
    
    return X_tau_values

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pyDOE import lhs
from scipy.integrate import odeint
import warnings
warnings.filterwarnings('ignore')

# Data size on the solution
nx, nt = (200, 200)
L = 1
T = 5
x = np.linspace(0, L, nx)
t = np.linspace(0, T, nt)

dx = L / (nx - 1)

x_grid, t_grid = np.meshgrid(x, t)

# Preparing the x and y together as an input for predictions in one single array, as x_star
data_all = np.hstack((x_grid.flatten()[:, None], t_grid.flatten()[:, None]))

fdm_N1_end = fdm_N1_pred[:, -1:].flatten()[:, None]
fdm_N2_end = fdm_N2_pred[:, -1:].flatten()[:, None]

fdm_N1 = fdm_N1_pred.flatten()[:, None]
fdm_N2 = fdm_N2_pred.flatten()[:, None]

# Determine how much points for training in both boundary condition and PDE itself
N_b = 50
N_u = 1500

# initial/boundary points
xx1 = np.vstack((x_grid[0, :].flatten(), t_grid[0, :].flatten())).T    # u(x, 0)
xx2 = np.vstack((x_grid[-1, :].flatten(), t_grid[-1, :].flatten())).T  # u(x, T)
xx3 = np.vstack((x_grid[:, 0].flatten(), t_grid[:, 0].flatten())).T    # u(0, t)
xx4 = np.vstack((x_grid[:, -1].flatten(), t_grid[:, -1].flatten())).T    # u(X, t)

xx = [xx1, xx2, xx3, xx4]
idx_b = [np.random.choice(x.shape[0], N_b, replace=False) for x in xx]

data_b_train_plot = [x[idx_b[j], :] for j, x in enumerate(xx)]

data_b_train = np.vstack([data_b_train_plot[0], data_b_train_plot[1], data_b_train_plot[2], data_b_train_plot[3]])

# Domain bounds (lowerbounds upperbounds) [x, t], which are here ([0, 0]) and ([1, 1])
lb = data_all.min(axis=0)
ub = data_all.max(axis=0)

# Preparing the training data inside PDE domain
data_u_train = lb + (ub-lb)*lhs(2, N_u)

idx = np.random.choice(data_u_train.shape[0], N_u, replace=False)

fdm_N1_train = fdm_N1[idx, :] 
fdm_N2_train = fdm_N2[idx, :] 

In [ ]:
import os
import time
import math
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense
from keras.models import Model, load_model
from keras.activations import tanh, relu

class PINN:
    def __init__(self, data_u_train, data_init, data_end, data_left, data_right, 
                 fdm_N1_train, fdm_N2_train, fdm_N1_end, fdm_N2_end, optimizer):
        self.data_u_train = data_u_train
        self.data_init = data_init
        self.data_end = data_end
        self.data_left = data_left
        self.data_right = data_right
        self.N1_fdm = tf.convert_to_tensor(fdm_N1_train, dtype=tf.float32)
        self.N2_fdm = tf.convert_to_tensor(fdm_N2_train, dtype=tf.float32)
        self.N1_fdm_end = tf.convert_to_tensor(fdm_N1_end, dtype=tf.float32)
        self.N2_fdm_end = tf.convert_to_tensor(fdm_N2_end, dtype=tf.float32)
        self.optimizer = optimizer
        self.build_model()

    def build_model(self):

        inputs = tf.keras.Input(shape=self.data_u_train.shape[1:])
        dense1 = Dense(50, activation='tanh')(inputs)
        dense2 = Dense(50, activation='tanh')(dense1)
        dense3 = Dense(50, activation='tanh')(dense2)
        dense4 = Dense(50, activation='tanh')(dense3)
        dense5 = Dense(50, activation='tanh')(dense4)
        outputs = Dense(2)(dense5)
        
        # Define the model
        self.model_nn = Model(inputs=inputs, outputs=outputs)
        self.model_nn.compile(optimizer=self.optimizer, loss=self.__loss)
        
        # Initialize the history list
        self.history = {"loss": [], "val_loss": []}
        
    def fit_model(self, epochs):
        earlystopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
        checpoint_cb = keras.callbacks.ModelCheckpoint('spde_pinn.h5', monitor='val_loss', save_best_only=True)
        history_callback = self.model_nn.fit(self.data_u_train, self.data_u_train, epochs=epochs, 
                                             verbose=1, validation_split=0.25)
        self.history["loss"].append(history_callback.history["loss"])
        self.history["val_loss"].append(history_callback.history["val_loss"])

    def u_model(self, data_train):        
        x_f = tf.convert_to_tensor(data_train[:, 0:1], dtype=tf.float32)
        t_f = tf.convert_to_tensor(data_train[:, 1:2], dtype=tf.float32)
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(x_f)
            tape.watch(t_f)
            with tf.GradientTape(persistent=True) as gtape:
                gtape.watch(x_f)
                gtape.watch(t_f)
                inp = tf.stack([x_f[:, 0], t_f[:, 0]], axis=1)
                u = self.model_nn(inp)
                N1 = u[:, :1]
                N2 = u[:, 1:2]
            N1_x = gtape.gradient(N1, x_f)
            N2_x = gtape.gradient(N2, x_f)
            
            N1_t = gtape.gradient(N1, t_f)
            N2_t = gtape.gradient(N2, t_f)
            del gtape
        N1_xx = tape.gradient(N1_x, x_f)
        N2_xx = tape.gradient(N2_x, x_f)
        del tape
        intervals = []
        X_tau_values = X_tau(t_f, intervals)

        D1, D2, r1, r2, a11, a12, a21, a22 = get_parameters()
        theta_1, theta_2 = get_theta(X_tau_values)
        
        f_N1 = N1_t - D1*N1_xx + r1*N1_x - (theta_1 - a11*N1)*N1 + a12*N1*N2
        f_N2 = N2_t - D2*N2_xx + r2*N2_x - (theta_2 - a22*N2)*N2 + a21*N1*N2
        
        return N1, N2, f_N1, f_N2

    def __loss(self, data_u_train, exact_u_train):

        data_init = self.data_init
        data_end = self.data_end
        data_left = self.data_left
        data_right = self.data_right
                
        N1_pred, N2_pred, f_N1_pred, f_N2_pred = self.u_model(self.data_u_train)
        
        # initial and terminal conditions
        init_N1 = self.model_nn(data_init)[:, :1]
        init_N2 = self.model_nn(data_init)[:, 1:2]
        
        loss_init_N1 = tf.reduce_mean(tf.square(init_N1 - data_init[:, :1]*(1 - data_init[:, :1])))
        loss_init_N2 = tf.reduce_mean(tf.square(init_N2 - (1/2)*data_init[:, :1]*(1 - data_init[:, :1])))
        
        # side boundary conditions
        left_N1 = self.model_nn(data_left)[:, :1]
        left_N2 = self.model_nn(data_left)[:, 1:2]
        
        right_N1 = self.model_nn(data_right)[:, :1]
        right_N2 = self.model_nn(data_right)[:, 1:2]
        
        b_N1 = self.model_nn(data_end)[:, :1]
        b_N2 = self.model_nn(data_end)[:, 1:2]
        
        loss_b_N1 = tf.reduce_mean(tf.square(left_N1)) + tf.reduce_mean(tf.square(right_N1)) + tf.reduce_mean(tf.square(b_N1 - self.N1_fdm_end))
        loss_b_N2 = tf.reduce_mean(tf.square(left_N2)) + tf.reduce_mean(tf.square(right_N2)) + tf.reduce_mean(tf.square(b_N2 - self.N2_fdm_end))
        
        loss_N1 = loss_init_N1 + loss_b_N1 + tf.reduce_mean(tf.square(f_N1_pred))
        loss_N2 = loss_init_N2 + loss_b_N2 + tf.reduce_mean(tf.square(f_N2_pred))
        return loss_N1 + loss_N2
    
    def predict(self, x):
        return self.model_nn.predict(x)
    
    keras.utils.get_custom_objects()["__loss"] = __loss

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

spde_pinn = PINN(data_u_train, xx1, xx2, xx3, xx4, fdm_N1_train, fdm_N2_train, fdm_N1_end, fdm_N2_end, optimizer)
spde_pinn.model_nn.summary()

In [ ]:
import time

start = time.perf_counter()
for i in range(1):
    print('cycle:------------------------------------', i+1)
    spde_pinn.fit_model(epochs=2000)
        
elapsed = time.perf_counter() - start
print('Elapsed %.3f seconds.' % elapsed)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 1, figsize=[5, 3])
axs.set_facecolor('lightgrey')
axs.grid(which='major', color='black', linestyle='--', linewidth=0.5)
axs.plot(spde_pinn.history["loss"][0], label='PINN Train Loss', color='black')
axs.plot(spde_pinn.history["val_loss"][0], label='PINN Val Loss', color='green')
axs.set_yscale('log')
axs.set_xlim([-50, 2050])
axs.set_ylim([2e-7, 2e-1])
axs.set_xlabel('Iter', fontsize=12, fontweight ='bold')
axs.set_ylabel('Loss value', fontsize=12, fontweight ='bold', x=0)
plt.legend()
plt.show()

In [ ]:
pinn_prediction = spde_pinn.predict(data_all)

In [ ]:
from numpy import format_float_scientific

index = spde_pinn.history["val_loss"][0].index(min(spde_pinn.history["val_loss"][0]))

print(format_float_scientific(spde_pinn.history["loss"][0][index], exp_digits=2, precision=3))
print(format_float_scientific(spde_pinn.history["val_loss"][0][index], exp_digits=2, precision=3))
print(len(spde_pinn.history["val_loss"][0]))
print('%.2f'%(elapsed/len(spde_pinn.history["val_loss"][0])))
print('%.2f'%elapsed)

In [ ]:
from scipy.interpolate import griddata

pinn_N1_pred = griddata(data_all, pinn_prediction[:, :1].flatten(), (x_grid, t_grid), method='cubic')
pinn_N2_pred = griddata(data_all, pinn_prediction[:, 1:2].flatten(), (x_grid, t_grid), method='cubic')

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import initializers
from keras import activations

class TSVDLayer(tf.keras.layers.Layer):
    def __init__(self, units, activation=None, use_bias=True, kernel_initializer="glorot_uniform", 
                 bias_initializer="zeros", **kwargs,):
        self.units = int(units) if not isinstance(units, int) else units
        if self.units < 0:
            raise ValueError(
                "Received an invalid value for `units`, expected "
                f"a positive integer. Received: units={units}"
            )
        self.rank = 5
        self.activation = activations.get(activation)
        self.use_bias = use_bias
        self.kernel_initializer = kernel_initializer
        self.bias_initializer = bias_initializer
        self.supports_masking = True
        super(TSVDLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        input_shape = tf.TensorShape(input_shape)
        last_dim = tf.compat.dimension_value(input_shape[-1])
        
        if self.use_bias:
            self.bias = self.add_weight("bias",shape=[self.units,],initializer=self.bias_initializer,trainable=True)
        else:
            self.bias = None
        #self.kernel = self.add_weight(name='kernel', shape=(last_dim, self.units), initializer=self.kernel_initializer, trainable=True)
        kernel = tf.random.normal(shape=(last_dim, self.units))
        #kernel = tf.tanh(kernel)
        s, u, v = tf.linalg.svd(kernel)
    
        s_r, U_r, V_r = tf.tanh(s[..., :self.rank]), tf.tanh(u[..., :, :self.rank]), tf.tanh(v[..., :, :self.rank])
        # Compute SVD of the weight matrices and sort the singular values
        
        self.U = self.add_weight(name='U', shape=(U_r.shape), initializer=initializers.Constant(U_r), trainable=True)
        self.V = self.add_weight(name='V', shape=(V_r.shape), initializer=initializers.Constant(V_r), trainable=True)
        self.S = self.add_weight(name='S', shape=(s_r.shape), initializer=initializers.Constant(s_r), trainable=True)
        super(TSVDLayer, self).build(input_shape)
    
    #@tf.function
    def call(self, x):
        # Compute output using SVD
        s_mat = tf.linalg.diag(self.S)
        outputs = tf.matmul(x, tf.matmul(self.U, tf.matmul(s_mat, self.V, adjoint_b=True)))
        # Apply activation function if specified
        if self.activation is not None:
            #outputs = tf.matmul(x, tf.matmul(self.U, tf.matmul(s_mat, self.V, adjoint_b=True)))
            outputs = self.activation(outputs)
            
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
            
        return outputs
    
    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "units": self.units,
                "activation": activations.serialize(self.activation),
                "use_bias": self.use_bias,
                "kernel_initializer": self.kernel_initializer,
                "bias_initializer": self.bias_initializer,
            }
        )
        return config
    
keras.utils.get_custom_objects()["TSVDLayer"] = TSVDLayer

In [ ]:
import os
import time
import math
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense
from keras.models import Model, load_model
from keras.activations import tanh, relu

class PINN:
    def __init__(self, data_u_train, data_init, data_end, data_left, data_right, 
                 fdm_N1_train, fdm_N2_train, fdm_N1_end, fdm_N2_end, optimizer):
        self.data_u_train = data_u_train
        self.data_init = data_init
        self.data_end = data_end
        self.data_left = data_left
        self.data_right = data_right
        self.N1_fdm = tf.convert_to_tensor(fdm_N1_train, dtype=tf.float32)
        self.N2_fdm = tf.convert_to_tensor(fdm_N2_train, dtype=tf.float32)
        self.N1_fdm_end = tf.convert_to_tensor(fdm_N1_end, dtype=tf.float32)
        self.N2_fdm_end = tf.convert_to_tensor(fdm_N2_end, dtype=tf.float32)
        self.optimizer = optimizer
        self.build_model()

    def build_model(self):
        #if os.path.exists('pinn_legendre.h5'):
        #    self.model_nn = load_model('pinn_legendre.h5')
        #else:
            
        inputs = tf.keras.Input(shape=self.data_u_train.shape[1:])
        dense1 = Dense(50, activation='tanh')(inputs)
        dense2 = Dense(50, activation='tanh')(dense1)
        dense3 = Dense(50, activation='tanh')(dense2)
        dense4 = Dense(50, activation='tanh')(dense3)
        TSVD_layer1 = TSVDLayer(50)(dense4)
        outputs = Dense(2)(TSVD_layer1)
        
        # Define the model
        self.model_nn = Model(inputs=inputs, outputs=outputs)
        self.model_nn.compile(optimizer=self.optimizer, loss=self.__loss)
        
        # Initialize the history list
        self.history = {"loss": [], "val_loss": []}
        
    def fit_model(self, epochs):
        earlystopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
        checpoint_cb = keras.callbacks.ModelCheckpoint('spde_pinn.h5', monitor='val_loss', save_best_only=True)
        history_callback = self.model_nn.fit(self.data_u_train, self.data_u_train, epochs=epochs, 
                                             callbacks=[checpoint_cb], verbose=1, validation_split=0.25)
        self.history["loss"].append(history_callback.history["loss"])
        self.history["val_loss"].append(history_callback.history["val_loss"])

    def u_model(self, data_train):        
        x_f = tf.convert_to_tensor(data_train[:, 0:1], dtype=tf.float32)
        t_f = tf.convert_to_tensor(data_train[:, 1:2], dtype=tf.float32)
        with tf.GradientTape(persistent=True) as tape:
            tape.watch(x_f)
            tape.watch(t_f)
            with tf.GradientTape(persistent=True) as gtape:
                gtape.watch(x_f)
                gtape.watch(t_f)
                inp = tf.stack([x_f[:, 0], t_f[:, 0]], axis=1)
                u = self.model_nn(inp)
                N1 = u[:, :1]
                N2 = u[:, 1:2]
            N1_x = gtape.gradient(N1, x_f)
            N2_x = gtape.gradient(N2, x_f)
            
            N1_t = gtape.gradient(N1, t_f)
            N2_t = gtape.gradient(N2, t_f)
            del gtape
        N1_xx = tape.gradient(N1_x, x_f)
        N2_xx = tape.gradient(N2_x, x_f)
        del tape
        intervals = []
        X_tau_values = X_tau(t_f, intervals)

        D1, D2, r1, r2, a11, a12, a21, a22 = get_parameters()
        theta_1, theta_2 = get_theta(X_tau_values)
        
        f_N1 = N1_t - D1*N1_xx + r1*N1_x - (theta_1 - a11*N1)*N1 + a12*N1*N2
        f_N2 = N2_t - D2*N2_xx + r2*N2_x - (theta_2 - a22*N2)*N2 + a21*N1*N2
        
        return N1, N2, f_N1, f_N2

    def __loss(self, data_u_train, exact_u_train):

        data_init = self.data_init
        data_end = self.data_end
        data_left = self.data_left
        data_right = self.data_right
                
        N1_pred, N2_pred, f_N1_pred, f_N2_pred = self.u_model(self.data_u_train)
        
        # initial and terminal conditions
        init_N1 = self.model_nn(data_init)[:, :1]
        init_N2 = self.model_nn(data_init)[:, 1:2]
        
        loss_init_N1 = tf.reduce_mean(tf.square(init_N1 - data_init[:, :1]*(1 - data_init[:, :1])))
        loss_init_N2 = tf.reduce_mean(tf.square(init_N2 - (1/2)*data_init[:, :1]*(1 - data_init[:, :1])))
        
        # side boundary conditions
        left_N1 = self.model_nn(data_left)[:, :1]
        left_N2 = self.model_nn(data_left)[:, 1:2]
        
        right_N1 = self.model_nn(data_right)[:, :1]
        right_N2 = self.model_nn(data_right)[:, 1:2]
        
        b_N1 = self.model_nn(data_end)[:, :1]
        b_N2 = self.model_nn(data_end)[:, 1:2]
        
        loss_b_N1 = tf.reduce_mean(tf.square(left_N1)) + tf.reduce_mean(tf.square(right_N1)) + tf.reduce_mean(tf.square(b_N1 - self.N1_fdm_end))
        loss_b_N2 = tf.reduce_mean(tf.square(left_N2)) + tf.reduce_mean(tf.square(right_N2)) + tf.reduce_mean(tf.square(b_N2 - self.N2_fdm_end))
        
        loss_N1 = loss_init_N1 + loss_b_N1 + tf.reduce_mean(tf.square(f_N1_pred))
        loss_N2 = loss_init_N2 + loss_b_N2 + tf.reduce_mean(tf.square(f_N2_pred))
        return loss_N1 + loss_N2
    
    def predict(self, x):
        return self.model_nn.predict(x)
    
    keras.utils.get_custom_objects()["__loss"] = __loss

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

spde_pinn = PINN(data_u_train, xx1, xx2, xx3, xx4, fdm_N1_train, fdm_N2_train, fdm_N1_end, fdm_N2_end, optimizer)
spde_pinn.model_nn.summary()

In [ ]:
import time

start = time.perf_counter()
for i in range(1):
    print('cycle:------------------------------------', i+1)
    spde_pinn.fit_model(epochs=2000)
    spde_pinn.model_nn.load_weights('spde_pinn.h5')
        
elapsed = time.perf_counter() - start
print('Elapsed %.3f seconds.' % elapsed)

Epoch 621/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.7172e-04 - val_loss: 2.6913e-04
Epoch 622/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.6002e-04 - val_loss: 2.6917e-04
Epoch 623/2000
36/36 [==============================] - 0s 10ms/step - loss: 3.0460e-04 - val_loss: 2.4372e-04
Epoch 624/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.4789e-04 - val_loss: 3.1193e-04
Epoch 625/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.5969e-04 - val_loss: 2.6645e-04
Epoch 626/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.9850e-04 - val_loss: 2.5976e-04
Epoch 627/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.7480e-04 - val_loss: 4.6633e-04
Epoch 628/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.7941e-04 - val_loss: 3.4211e-04
Epoch 629/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.5493e-04 - val_loss: 2.5365e-04
E

Epoch 695/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.9275e-04 - val_loss: 1.6628e-04
Epoch 696/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.8860e-04 - val_loss: 1.9325e-04
Epoch 697/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.3370e-04 - val_loss: 1.9068e-04
Epoch 698/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.8168e-04 - val_loss: 1.8668e-04
Epoch 699/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.0763e-04 - val_loss: 1.7439e-04
Epoch 700/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.9665e-04 - val_loss: 2.3721e-04
Epoch 701/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.1355e-04 - val_loss: 2.3444e-04
Epoch 702/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.8724e-04 - val_loss: 2.1478e-04
Epoch 703/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.9270e-04 - val_loss: 1.6479e-04
E

Epoch 769/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.8415e-04 - val_loss: 1.1877e-04
Epoch 770/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.1566e-04 - val_loss: 1.1503e-04
Epoch 771/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.9256e-04 - val_loss: 1.3010e-04
Epoch 772/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.3762e-04 - val_loss: 2.1677e-04
Epoch 773/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.4971e-04 - val_loss: 1.2798e-04
Epoch 774/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.4357e-04 - val_loss: 1.1476e-04
Epoch 775/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.4430e-04 - val_loss: 1.4429e-04
Epoch 776/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.5804e-04 - val_loss: 1.2323e-04
Epoch 777/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.5187e-04 - val_loss: 1.2819e-04
E

Epoch 843/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.5187e-04 - val_loss: 8.2523e-04
Epoch 844/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.9129e-04 - val_loss: 9.4094e-05
Epoch 845/2000
36/36 [==============================] - 0s 11ms/step - loss: 8.0812e-05 - val_loss: 7.7655e-05
Epoch 846/2000
36/36 [==============================] - 0s 10ms/step - loss: 7.6987e-05 - val_loss: 7.6622e-05
Epoch 847/2000
36/36 [==============================] - 0s 10ms/step - loss: 7.6384e-05 - val_loss: 7.6142e-05
Epoch 848/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.6170e-04 - val_loss: 1.1879e-04
Epoch 849/2000
36/36 [==============================] - 0s 10ms/step - loss: 8.8934e-05 - val_loss: 7.7904e-05
Epoch 850/2000
36/36 [==============================] - 0s 10ms/step - loss: 9.6800e-05 - val_loss: 2.6252e-04
Epoch 851/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.1556e-04 - val_loss: 7.8704e-05
E

Epoch 917/2000
36/36 [==============================] - 0s 11ms/step - loss: 5.9030e-05 - val_loss: 5.7018e-05
Epoch 918/2000
36/36 [==============================] - 0s 11ms/step - loss: 5.6827e-05 - val_loss: 5.6606e-05
Epoch 919/2000
36/36 [==============================] - 0s 10ms/step - loss: 6.5944e-05 - val_loss: 3.0730e-04
Epoch 920/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.3824e-04 - val_loss: 6.4962e-05
Epoch 921/2000
36/36 [==============================] - 0s 11ms/step - loss: 5.8346e-05 - val_loss: 5.6215e-05
Epoch 922/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.1581e-04 - val_loss: 1.4755e-04
Epoch 923/2000
36/36 [==============================] - 0s 10ms/step - loss: 8.4269e-05 - val_loss: 5.6097e-05
Epoch 924/2000
36/36 [==============================] - 0s 10ms/step - loss: 8.9548e-05 - val_loss: 1.1805e-04
Epoch 925/2000
36/36 [==============================] - 0s 11ms/step - loss: 6.5459e-05 - val_loss: 5.5786e-05
E

...

36/36 [==============================] - 0s 10ms/step - loss: 1.4874e-05 - val_loss: 1.4437e-05
Epoch 1649/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.4150e-05 - val_loss: 1.3873e-05
Epoch 1650/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.3678e-05 - val_loss: 1.3486e-05
Epoch 1651/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.3344e-05 - val_loss: 1.3201e-05
Epoch 1652/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.3092e-05 - val_loss: 1.2982e-05
Epoch 1653/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.2895e-05 - val_loss: 1.2806e-05
Epoch 1654/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.2735e-05 - val_loss: 1.2661e-05
Epoch 1655/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.2601e-05 - val_loss: 1.2538e-05
Epoch 1656/2000
36/36 [==============================] - 0s 11ms/step - loss: 1.2485e-05 - val_loss: 1.2430e-05
Epoch 16

36/36 [==============================] - 0s 10ms/step - loss: 3.1281e-05 - val_loss: 1.2015e-05
Epoch 1722/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.4832e-05 - val_loss: 2.2082e-05
Epoch 1723/2000
36/36 [==============================] - 0s 10ms/step - loss: 4.0849e-05 - val_loss: 6.5744e-05
Epoch 1724/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.4695e-05 - val_loss: 1.2147e-05
Epoch 1725/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.0843e-05 - val_loss: 1.0358e-05
Epoch 1726/2000
36/36 [==============================] - 0s 10ms/step - loss: 3.5689e-05 - val_loss: 8.7390e-05
Epoch 1727/2000
36/36 [==============================] - 0s 10ms/step - loss: 3.1793e-05 - val_loss: 1.3964e-05
Epoch 1728/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.7079e-05 - val_loss: 4.0266e-05
Epoch 1729/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.7284e-05 - val_loss: 2.4621e-05
Epoch 17

36/36 [==============================] - 0s 10ms/step - loss: 9.9728e-06 - val_loss: 9.9138e-06
Epoch 1795/2000
36/36 [==============================] - 0s 10ms/step - loss: 9.8637e-06 - val_loss: 9.8100e-06
Epoch 1796/2000
36/36 [==============================] - 0s 11ms/step - loss: 9.7641e-06 - val_loss: 9.7148e-06
Epoch 1797/2000
36/36 [==============================] - 0s 10ms/step - loss: 9.6726e-06 - val_loss: 9.6277e-06
Epoch 1798/2000
36/36 [==============================] - 0s 11ms/step - loss: 6.0307e-05 - val_loss: 1.2415e-04
Epoch 1799/2000
36/36 [==============================] - 0s 10ms/step - loss: 4.6394e-05 - val_loss: 1.3964e-05
Epoch 1800/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.0488e-05 - val_loss: 9.5107e-06
Epoch 1801/2000
36/36 [==============================] - 0s 11ms/step - loss: 9.4420e-06 - val_loss: 9.3848e-06
Epoch 1802/2000
36/36 [==============================] - 0s 11ms/step - loss: 9.3508e-06 - val_loss: 9.3160e-06
Epoch 18

36/36 [==============================] - 0s 11ms/step - loss: 4.1777e-05 - val_loss: 1.3172e-05
Epoch 1868/2000
36/36 [==============================] - 0s 11ms/step - loss: 1.3762e-05 - val_loss: 8.9886e-06
Epoch 1869/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.8814e-05 - val_loss: 1.0698e-05
Epoch 1870/2000
36/36 [==============================] - 0s 10ms/step - loss: 4.0690e-05 - val_loss: 1.6419e-05
Epoch 1871/2000
36/36 [==============================] - 0s 11ms/step - loss: 1.2667e-05 - val_loss: 8.1652e-06
Epoch 1872/2000
36/36 [==============================] - 0s 10ms/step - loss: 8.9889e-06 - val_loss: 2.5718e-05
Epoch 1873/2000
36/36 [==============================] - 0s 10ms/step - loss: 6.6589e-05 - val_loss: 1.5298e-05
Epoch 1874/2000
36/36 [==============================] - 0s 11ms/step - loss: 1.0637e-05 - val_loss: 7.8273e-06
Epoch 1875/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.2522e-05 - val_loss: 9.8735e-05
Epoch 18

36/36 [==============================] - 0s 10ms/step - loss: 9.2180e-06 - val_loss: 9.1736e-06
Epoch 1941/2000
36/36 [==============================] - 0s 10ms/step - loss: 9.1355e-06 - val_loss: 9.0944e-06
Epoch 1942/2000
36/36 [==============================] - 0s 10ms/step - loss: 9.0588e-06 - val_loss: 9.0205e-06
Epoch 1943/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.3068e-05 - val_loss: 1.2260e-05
Epoch 1944/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.1627e-05 - val_loss: 1.7871e-05
Epoch 1945/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.1497e-05 - val_loss: 3.1331e-05
Epoch 1946/2000
36/36 [==============================] - 0s 11ms/step - loss: 2.1186e-05 - val_loss: 4.8989e-05
Epoch 1947/2000
36/36 [==============================] - 0s 10ms/step - loss: 2.0065e-05 - val_loss: 2.4405e-05
Epoch 1948/2000
36/36 [==============================] - 0s 10ms/step - loss: 1.2666e-05 - val_loss: 2.5458e-05
Epoch 19

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 1, figsize=[5, 3])
axs.set_facecolor('lightgrey')
axs.grid(which='major', color='black', linestyle='--', linewidth=0.5)
axs.plot(spde_pinn.history["loss"][0], label='PINN-TSVD Train Loss', color='black')
axs.plot(spde_pinn.history["val_loss"][0], label='PINN-TSVD Val Loss', color='orange')
axs.set_yscale('log')
axs.set_xlim([-50, 2050])
axs.set_ylim([2e-7, 2e-1])
axs.set_xlabel('Iter', fontsize=12, fontweight ='bold')
axs.set_ylabel('Loss value', fontsize=12, fontweight ='bold', x=0)
plt.legend()
plt.show()

In [ ]:
pinn_tsvd_prediction = spde_pinn.predict(data_all)

In [ ]:
from numpy import format_float_scientific

index = spde_pinn.history["val_loss"][0].index(min(spde_pinn.history["val_loss"][0]))

print(format_float_scientific(spde_pinn.history["loss"][0][index], exp_digits=2, precision=3))
print(format_float_scientific(spde_pinn.history["val_loss"][0][index], exp_digits=2, precision=3))
print(len(spde_pinn.history["val_loss"][0]))
print('%.2f'%(elapsed/len(spde_pinn.history["val_loss"][0])))
print('%.2f'%elapsed)

In [ ]:
from scipy.interpolate import griddata

pinn_tsvd_N1_pred = griddata(data_all, pinn_tsvd_prediction[:, :1].flatten(), (x_grid, t_grid), method='cubic')
pinn_tsvd_N2_pred = griddata(data_all, pinn_tsvd_prediction[:, 1:2].flatten(), (x_grid, t_grid), method='cubic')

In [ ]:
# Plot species u
fig = plt.figure(figsize=[10, 5])

vmin = min(pinn_tsvd_N1_pred.min(), pinn_tsvd_N2_pred.min())
vmax = max(pinn_tsvd_N1_pred.max(), pinn_tsvd_N2_pred.max())

ax1 = fig.add_subplot(121)
cax = ax1.contourf(x_grid, t_grid, pinn_tsvd_N1_pred, 50, cmap='jet', vmin=vmin, vmax=vmax)
ax1.set_xlabel('Spatial coordinate (x)')
ax1.set_ylabel('Time (t)')
ax1.set_title('Population Density of Native Species $N_1$')
fig.colorbar(cax, ax=ax1, location='right')

# Plot species v
ax2 = fig.add_subplot(122)
cbx = ax2.contourf(x_grid, t_grid, pinn_tsvd_N2_pred, 50, cmap='jet', vmin=vmin, vmax=vmax)
#ax2.colorbar(label='Population density')
ax2.set_xlabel('Spatial coordinate (x)')
ax2.set_title('Population Density of Invasive Species $N_2$')

fig.colorbar(cbx, ax=ax2, location='right')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.gridspec import GridSpec
from matplotlib import ticker

u_fdm = fdm_N1_pred.T
v_fdm = fdm_N2_pred.T

u_pinn = pinn_N1_pred
v_pinn = pinn_N2_pred

u_pinn_tsvd = pinn_tsvd_N1_pred
v_pinn_tsvd = pinn_tsvd_N2_pred

# Create figure with GridSpec (3 rows: contour plots, small plots, and small plots)
fig = plt.figure(figsize=(12, 6))
gs = GridSpec(2, 3, height_ratios=[3, 2], hspace=0.4, wspace=0.4)

# Define data and corresponding titles
data_fdm = [u_fdm, v_fdm]
data_pinn = [u_pinn, v_pinn]
data_pinn_tsvd = [u_pinn_tsvd, v_pinn_tsvd]
titles = [r'$N_1(x, t)$ PINN-TSVD solution', r'$N_2(x, y)$ PINN-TSVD solution']

cl = []

for i, (data, title) in enumerate(zip(data_pinn_tsvd, titles)):
    ax = fig.add_subplot(gs[0, i])  # Assign subplot in the first row
    c = ax.contourf(x_grid, t_grid, data, 50, cmap='jet', vmin=0, vmax=0.7)  # Independent colormap for each plot
    cl.append(c)  # Store contour plot for colorbar
    ax.set_title(title, fontsize=12, fontweight='bold')  # Set title
    ax.axvline(x=x[120], color='k', linestyle='--', linewidth=1)  # Add vertical line
    
    # Add default colorbar for each subplot
    fig.colorbar(c, ax=ax)  # Uses default placement
    ax.set_xlabel('x', fontsize=12, fontweight='bold')

    # **Set ylabel only for the rightmost plot**
    if i == 0:
        ax.set_ylabel('t', fontsize=12, fontweight='bold', labelpad=10)
    
# Define data and corresponding titles for line plots
titles = [r'$N_1(x, t)$ at $x=0.6$', r'$N_2(x, t)$ at $x=0.6$']

fig.subplots_adjust(wspace=0.3)
# Create subplots with iteration
for i, (fdm, pinn, pinn_tsvd, title) in enumerate(zip(data_fdm, data_pinn, data_pinn_tsvd, titles)):
    ax = fig.add_subplot(gs[1, i])  # Assign subplot in the second row
    ax.set_facecolor('lightgrey')  # Set background color
    ax.grid(which='major', color='black', linestyle='--', linewidth=0.5)  # Grid settings
        
    # Plot PINN solution
    ax.plot(t, fdm[:, 120].reshape(-1, 1), lw=6, ls='-', label='FDM')
    ax.plot(t, pinn[:, 120].reshape(-1, 1), lw=5, ls='--', label='PINN')
    ax.plot(t, pinn_tsvd[:, 120].reshape(-1, 1), lw=3, ls=':', color='yellow', label='PINN-TSVD')
    # Set title dynamically based on x-value
    ax.set_title(title.format(x[25]), fontsize=12)
    ax.set_ylim([0, 0.9])  # Set Y-axis limits
    ax.set_xlabel('t', fontsize=12, fontweight='bold')

    if i == 0:
        ax.set_ylabel('$N_1(x, y)$', fontsize=12, fontweight='bold', labelpad=10)
    
    if i == 1:
        ax.set_ylabel('$N_2(x, y)$', fontsize=12, fontweight='bold', labelpad=10)
        #ax.set_ylim([0.08, 0.1])  # Set Y-axis limits
        ax.legend(loc='center', bbox_to_anchor=(-0.2, -0.45), ncol=5, frameon=False)
    ax.grid(True)  # Enable grid